# Time-resolved recovery

Applies the same principle along the time axis: the in-situ hybridization series
is split into non-overlapping windows of consecutive timepoints, and each window
is recovered independently, giving one spectrum and one coefficient pair per
window.

**Input** — a folder of time-series CSVs, one per concentration, plus the
DNA-probe and measured RNA references. Window sizes are set in `window_configs`.

**Output** — one folder per window, with the recovered spectrum, coefficients and
metrics, plus post-processing cells that assemble the per-window results into
spectrum-versus-time and metric-versus-time summaries.

**Feeds** — Fig. 5 and Supplementary Figs. S6–S12.

**Scale factor.** Spectra are multiplied by `SCALE_FACTOR = 400` before the network sees them and the factor is divided out again before anything is written, so every saved spectrum is on the mean-normalized scale. This is a numerical-conditioning choice only: the measured spectrum, the DNA reference and the ground-truth reference are all scaled by the same constant, so the recovered coefficients and every shape-based metric are unchanged.


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import torch.nn.functional as F
import torch.optim as optim
import os
import matplotlib.pyplot as plt
from scipy.spatial.distance import cosine
from sklearn.metrics import r2_score
from scipy.stats import pearsonr
from pathlib import Path

In [ ]:
# """
# ULTIMATE VERSION: Process multiple CSVs with multiple (N,M) window configurations
# """
META = {
    'bg_path': '/mnt/f/Jiaheng Cui/DNA RNA hybridization/Data/12192025-Time_dependent_N_gene_despiked/ProbeDNA-Avg(reference).csv',
    'true_path': f'/mnt/f/Jiaheng Cui/DNA RNA hybridization/Data/12192025-Time_dependent_N_gene_despiked/SARSCoV2ref1(true).csv',
    'pos_folder': '/mnt/f/Jiaheng Cui/DNA RNA hybridization/Results/12192025-Yingchuan_scaled_code_time_dependent/Unnormalized_data',
    'bg_sep': '\t', 
    'output_path': f'/mnt/f/Jiaheng Cui/DNA RNA hybridization/Results/12192025-Yingchuan_scaled_code_time_dependent/Unnormalized_data/05212026-1E7-only_mean_norm-extraction',
    
    'csv_list': ["05212026-3-N gene-time_mean_norm.csv"], # Option 2: Set to a list of specific csv filenames to process only those,e.g., ["1.csv", "3.csv", "4.csv"]

    # Window configurations to test:
    # List of (N, M) tuples where N=window_size, M=step_size
    'window_configs': [
        (3, 3)
    ],
}

os.makedirs(META['output_path'], exist_ok=True)
SCALE_FACTOR = 400.0

In [ ]:
# Get list of CSV files to process
pos_folder = Path(META['pos_folder'])
if META['csv_list'] is None:
    csv_files = sorted([f for f in pos_folder.glob('*.csv')])
    print(f"Found {len(csv_files)} CSV files to process")
else:
    csv_files = [pos_folder / fname for fname in META['csv_list']]
    csv_files = [f for f in csv_files if f.exists()]
    print(f"Processing {len(csv_files)} specified CSV files")

if len(csv_files) == 0:
    raise ValueError("No CSV files found to process!")

print("\nFiles to process:")
for f in csv_files:
    print(f"  - {f.name}")

print(f"\nWindow configurations to test: {len(META['window_configs'])}")
for n, m in META['window_configs']:
    print(f"  - N={n}, M={m}")

total_jobs = len(csv_files) * len(META['window_configs'])
print(f"\nTotal combinations to process: {total_jobs}")

In [ ]:
def load_data(pos_path, bg_path, true_path, bg_sep=',', time_indices=None):
    """
    Load data with optional time window selection.
    
    Args:
        pos_path: Path to mixture CSV
        bg_path: Path to background CSV
        true_path: Path to true signal CSV
        bg_sep: Separator for background CSV
        time_indices: List of column indices to use (1-indexed, excludes x column).
                     If None, use all columns.
    """
    # Background
    bg_spectra = pd.read_csv(bg_path, sep=bg_sep)
    bg_mean = bg_spectra.iloc[:, 1].tolist()
    bg_mean_mu = np.mean(bg_mean)
    if bg_mean_mu != 0:
        bg_mean = bg_mean / bg_mean_mu
    else:
        print("[Warning] Background mean is zero!")
    
    # True
    true_data = pd.read_csv(true_path)
    true_y = true_data.iloc[:, 1].values.astype(np.float32)
    true_mu = true_y.mean()
    if true_mu != 0:
        true_y = true_y / true_mu
    else:
        print("[Warning] True signal mean is zero!")
    
    # Positive
    base_data = pd.read_csv(pos_path)
    
    # If time_indices specified, select only those columns
    if time_indices is not None:
        selected_cols = [0] + [idx for idx in time_indices]
        base_data = base_data.iloc[:, selected_cols]
    
    col_x = base_data.columns[0]
    signal_cols = list(base_data.columns[1:])
    
    parsed_vals = []
    convertible = True
    for c in signal_cols:
        if isinstance(c, str):
            try:
                parsed_vals.append(float(c))
            except ValueError:
                convertible = False
                break
        else:
            parsed_vals.append(float(c))
    
    if convertible:
        parsed_vals = np.array(parsed_vals, dtype=np.float32)
        new_signal_cols = parsed_vals
        vmin, vmax = parsed_vals.min(), parsed_vals.max()
        if vmax > vmin:
            new_signal_cols = (parsed_vals - vmin) / (vmax - vmin)
        else:
            print("[Warning] Signal column values are constant, using linspace.")
            new_signal_cols = np.linspace(0, 1, len(signal_cols))
    else:
        new_signal_cols = np.linspace(0, 1, len(signal_cols))
    
    base_data.columns = [col_x] + list(new_signal_cols)
    
    # Mean normalization
    signal_df = base_data.iloc[:, 1:].astype(np.float32)
    col_means = signal_df.mean(axis=0)
    if (col_means == 0).any():
        print("[Warning] Some columns have zero mean!")
    else:
        base_data.iloc[:, 1:] = signal_df / col_means
    
    base_data.rename(columns={base_data.columns[0]: "x"}, inplace=True)
    df_long = base_data.melt(id_vars=["x"], var_name="s", value_name="y")
    
    return df_long, bg_mean, true_y

In [ ]:
class FourierFeatureMapping(nn.Module):
    def __init__(self, num_frequencies=6, include_input=True):
        super().__init__()
        self.num_frequencies = num_frequencies
        self.include_input = include_input
        self.freq_bands = 2.0 ** torch.arange(0, num_frequencies).float() * np.pi
    
    def forward(self, x):
        out = [x] if self.include_input else []
        for freq in self.freq_bands.to(x.device):
            out.append(torch.sin(freq * x))
            out.append(torch.cos(freq * x))
        return torch.cat(out, dim=-1)

class ResBlock(nn.Module):
    def __init__(self, dim):
        super(ResBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )
        self.activation = nn.ReLU()
    
    def forward(self, x):
        return self.activation(x + self.block(x))

class SERSDecomposition(nn.Module):
    def __init__(self, input_x_dim=13, input_s_dim=1, hidden_dim=256, z_dim=128): 
        super(SERSDecomposition, self).__init__()
        
        # f(x) branch
        self.f_input = nn.Sequential(
            nn.Linear(input_x_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )   
        self.f_blocks = nn.Sequential(
            ResBlock(hidden_dim),
            ResBlock(hidden_dim)
        )
        self.f_output = nn.Sequential(
            nn.Linear(hidden_dim, z_dim),
            nn.ReLU(),
            nn.Linear(z_dim, 1)
        )
        
        self.c_input = nn.Sequential(
            nn.Linear(input_s_dim, hidden_dim),
            nn.ReLU()
        )
        self.c_blocks = nn.Sequential(
            ResBlock(hidden_dim),
            ResBlock(hidden_dim),
            ResBlock(hidden_dim)
        )
        self.c_output = nn.Sequential(
            nn.Linear(hidden_dim, 2),
            nn.Softplus() 
        )
    
    def forward(self, x_embed, s):
        fx = self.f_input(x_embed)
        fx = self.f_blocks(fx)
        f_x = self.f_output(fx)
        
        cs = self.c_input(s)
        cs = self.c_blocks(cs)
        a_c = self.c_output(cs)
        a_s = a_c[:, 0:1]
        b_s = a_c[:, 1:2]
        
        return a_s, b_s, f_x

def weights_init(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

In [ ]:
def flatness_penalty(f_x, threshold, region_length=5):
    f_x = f_x.view(-1)
    diffs = f_x[1:] - f_x[:-1]
    sq_diffs = diffs ** 2
    sq_diffs = sq_diffs.unsqueeze(0).unsqueeze(0)
    kernel = torch.ones(1, 1, region_length, device=f_x.device) / region_length
    avg_sq = F.conv1d(sq_diffs, kernel, padding=region_length // 2).squeeze()
    penalty = torch.clamp(threshold - avg_sq, min=0.0)
    return torch.mean(penalty)

def custom_loss(y_true, a_s, b_s, f_x, bg, scale_mode=SCALE_FACTOR, 
                lambda_penalty=1.0, lambda_flat=0.05, threshold=1e-3, apply_flatness=False):
    
    threshold = threshold * (scale_mode ** 2)
    recon = a_s * f_x + b_s * bg
    mse_loss = torch.mean((y_true - recon) ** 2)
    neg_penalty = torch.mean(torch.clamp(-f_x, min=0.0)) * scale_mode 
    
    flat_pen = 0.0
    if apply_flatness:
        flat_pen = flatness_penalty(f_x, region_length=5, threshold=threshold)
    
    total_loss = (mse_loss + 
                  lambda_penalty * neg_penalty + 
                  lambda_flat * flat_pen)
                  
    return total_loss

In [ ]:
def to_1d(x):
    return x.detach().cpu().numpy().reshape(-1)

def train_single_window(pos_path, bg_path, true_path, output_dir, time_indices, 
                        window_name, bg_sep=',', scale_factor=400.0, 
                        num_epochs=1000, device='cuda', verbose=True):
    """
    Train model on a single time window and save results to output_dir
    """
    if verbose:
        print(f"\n{'='*60}")
        print(f"Processing window: {window_name}")
        print(f"Time indices: {time_indices}")
        print(f"Output to: {output_dir}")
        print(f"{'='*60}\n")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Load data for this window
    dt, bg_mean_spectrum, true_y = load_data(pos_path, bg_path, true_path, bg_sep, time_indices)
    
    # Prepare Data
    x = dt[['x']].values.astype(np.float32)
    y = dt[['y']].values.astype(np.float32)
    s_raw = dt['s'].values
    s_train = torch.tensor(s_raw.astype(np.float32).reshape(-1, 1)).to(device)
    
    bg = np.tile(bg_mean_spectrum, (len(y)//len(bg_mean_spectrum), 1)).reshape(-1, 1)
    x_train = torch.tensor(x, dtype=torch.float32).to(device)
    y_train = torch.tensor(y, dtype=torch.float32).to(device)
    bg_train = torch.tensor(bg, dtype=torch.float32).to(device)
    
    # Scaling
    y_train = y_train * scale_factor
    bg_train = bg_train * scale_factor
    
    # Normalize x
    x_train = (x_train - x_train.min()) / (x_train.max() - x_train.min())
    
    # Feature Mapping
    fourier_mapping = FourierFeatureMapping(num_frequencies=6).to(device)
    x_embed_train = fourier_mapping(x_train)
    
    # Model Init
    input_x_dim = x_embed_train.shape[1]
    model = SERSDecomposition(input_x_dim=input_x_dim).to(device)
    model.apply(weights_init)
    
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1000, gamma=0.5)
    
    # Loop variables
    loss_history = []
    f_final_list, g_final_list, h_final_list = [], [], []
    r2_recon_list, cos_recon_list = [], []
    metrics_extracted = {
        "r2": [],
        "cosine": [], 
        "pearson": []
    }
    
    # Training loop
    for epoch in range(num_epochs):
        model.train()
        apply_flatness = epoch >= 500
        
        a_s, b_s, f_x = model(x_embed_train, s=s_train)
        
        loss = custom_loss(
            y_train, a_s, b_s, f_x, bg_train, 
            scale_mode=scale_factor,
            apply_flatness=apply_flatness 
        )
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5)
        optimizer.step()
        scheduler.step()
        
        loss_history.append(loss.item())
        
        # Post-processing
        s_train_np = to_1d(s_train)
        bg_train_np = to_1d(bg_train)
        f_x_np = to_1d(f_x)
        a_s_np = to_1d(a_s)
        b_s_np = to_1d(b_s)
        y_train_np = to_1d(y_train)
        x_vals = dt['x'].values.astype(np.float32)
        
        unique_s = np.unique(s_train_np)
        
        mask0 = (s_train_np == unique_s[0]).squeeze()
        f0 = f_x_np[mask0]
        bg0 = bg_train_np[mask0]
        x0 = x_vals[mask0]
        
        # Area normalization
        f_area = np.trapz(f0, x=x0, axis=0)
        bg_area = np.trapz(bg0, x=x0, axis=0)
        
        f_final = f0 * (bg_area / f_area)
        bg_final = bg0
        true_final = true_y * scale_factor
        
        f_final_list.append(f_final)
        metrics_extracted["r2"].append(r2_score(true_final, f_final))
        metrics_extracted["cosine"].append(1 - cosine(true_final, f_final))
        pearson_corr, _ = pearsonr(true_final, f_final)
        metrics_extracted["pearson"].append(pearson_corr)
        
        a_concs, b_concs = [], []
        r2_concs, cos_concs = [], []
        
        for s_val in unique_s:
            mask = (s_train_np == s_val).squeeze()
            y_sub = y_train_np[mask]
            
            a_pred_sub = a_s_np[mask][0]
            b_pred_sub = b_s_np[mask][0]
            
            a_final = a_pred_sub * (f_area / bg_area)
            b_final = b_pred_sub
            
            a_concs.append(a_final)
            b_concs.append(b_final)
            
            recon_np = a_final * f_final + b_final * bg_final
            r2_concs.append(r2_score(y_sub, recon_np))
            cos_concs.append(1 - cosine(y_sub, recon_np))
        
        g_final_list.append(np.array(a_concs))
        h_final_list.append(np.array(b_concs))
        r2_recon_list.append(np.array(r2_concs))
        cos_recon_list.append(np.array(cos_concs))
        
        # Progress update
        if verbose and (epoch + 1) % 200 == 0:
            print(f"Epoch {epoch+1}/{num_epochs} | Loss: {loss.item():.6f} | "
                  f"R2: {metrics_extracted['r2'][-1]:.4f}")
    
    if verbose:
        print(f"Training completed for {window_name}!")
    
    # Return all results
    return {
        'x0': x0,
        'f_final_list': f_final_list,
        'g_final_list': g_final_list,
        'h_final_list': h_final_list,
        'r2_recon_list': r2_recon_list,
        'cos_recon_list': cos_recon_list,
        'metrics_extracted': metrics_extracted,
        'unique_s': unique_s,
        's_train_np': s_train_np,
        'y_train_np': y_train_np,
        'bg_final': bg_final,
        'true_final': true_final,
        'loss_history': loss_history,
        'time_indices': time_indices
    }

In [ ]:
def save_outputs(results, output_dir, verbose=True):
    """
    Save all outputs (CSV and figures) to output_dir
    """
    x0 = results['x0']
    f_final_list = results['f_final_list']
    g_final_list = results['g_final_list']
    h_final_list = results['h_final_list']
    r2_recon_list = results['r2_recon_list']
    cos_recon_list = results['cos_recon_list']
    metrics_extracted = results['metrics_extracted']
    unique_s = results['unique_s']
    s_train_np = results['s_train_np']
    y_train_np = results['y_train_np']
    bg_final = results['bg_final']
    true_final = results['true_final']
    loss_history = results['loss_history']
    
    num_epochs = len(f_final_list)
    num_mixtures = len(unique_s)
    epochs = np.arange(1, num_epochs + 1)
    mixture_names = [f"mixture_{i}" for i in range(num_mixtures)]
    
    # 1. Extracted spectrum vs epoch
    df_f = pd.DataFrame(
        np.column_stack([x0] + [f / SCALE_FACTOR for f in f_final_list]),  # undo the x400 conditioning factor: saved spectra are mean-normalized
        columns=["Wavenumbers"] + [f"Epoch_{i}" for i in epochs]
    )
    df_f.to_csv(os.path.join(output_dir, "extracted_spectrum_vs_epoch.csv"), index=False)
    
    # 2. a (concentration) vs epoch
    df_a = pd.DataFrame(
        np.vstack(g_final_list),
        columns=mixture_names
    )
    df_a.insert(0, "Epoch", epochs)
    df_a.to_csv(os.path.join(output_dir, "g_vs_epoch.csv"), index=False)
    
    # 3. b (background coef) vs epoch
    df_b = pd.DataFrame(
        np.vstack(h_final_list),
        columns=mixture_names
    )
    df_b.insert(0, "Epoch", epochs)
    df_b.to_csv(os.path.join(output_dir, "h_vs_epoch.csv"), index=False)
    
    # 4. R2: reconstructed vs experimental
    df_r2_recon = pd.DataFrame(
        np.vstack(r2_recon_list),
        columns=mixture_names
    )
    df_r2_recon.insert(0, "Epoch", epochs)
    df_r2_recon.to_csv(os.path.join(output_dir, "r2_recon_vs_epoch.csv"), index=False)
    
    # 5. Cosine similarity: reconstructed vs experimental
    df_cos_recon = pd.DataFrame(
        np.vstack(cos_recon_list),
        columns=mixture_names
    )
    df_cos_recon.insert(0, "Epoch", epochs)
    df_cos_recon.to_csv(os.path.join(output_dir, "cosine_recon_vs_epoch.csv"), index=False)
    
    # 6. Extracted vs true metrics
    df_extract_metrics = pd.DataFrame(
        [
            metrics_extracted["r2"],
            metrics_extracted["cosine"],
            metrics_extracted["pearson"],
        ],
        index=["R2", "Cosine", "Pearson"],
        columns=[f"Epoch_{i}" for i in epochs]
    )
    df_extract_metrics.to_csv(
        os.path.join(output_dir, "extracted_vs_true_metrics.csv")
    )
    
    # 7. Figure i: Experimental vs reconstructed (each mixture)
    final_epoch_idx = -1
    
    for i, s_val in enumerate(unique_s):
        mask = (s_train_np == s_val)
        y_exp = y_train_np[mask]
        
        recon = (
            g_final_list[final_epoch_idx][i] * f_final_list[final_epoch_idx]
            + h_final_list[final_epoch_idx][i] * bg_final
        )
        
        plt.figure(figsize=(6, 4))
        plt.plot(x0, y_exp, label="Experimental", lw=1)
        plt.plot(x0, recon, label="Reconstructed", lw=1)
        
        plt.title(
            f"{mixture_names[i]} | "
            f"R2={r2_recon_list[final_epoch_idx][i]:.3f}, "
            f"Cos={cos_recon_list[final_epoch_idx][i]:.3f}"
        )
        plt.legend()
        plt.tight_layout()
        
        fname = f"fig_mixture_{i}_reconstruction.png"
        plt.savefig(os.path.join(output_dir, fname), dpi=300)
        plt.close()
    
    # 8. Figure ii: Extracted vs true spectrum
    plt.figure(figsize=(6, 4))
    plt.plot(x0, true_final, label="True", lw=2)
    plt.plot(x0, f_final_list[final_epoch_idx], label="Extracted", lw=2)
    
    plt.title(
        f"Extracted vs True | "
        f"R2={metrics_extracted['r2'][final_epoch_idx]:.3f}, "
        f"Cos={metrics_extracted['cosine'][final_epoch_idx]:.3f}, "
        f"Pearson={metrics_extracted['pearson'][final_epoch_idx]:.3f}"
    )
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "fig_extracted_vs_true.png"), dpi=300)
    plt.close()
    
    if verbose:
        print(f"  Outputs saved to: {output_dir}")

    # # 9. Loss vs extraction step
    # df_loss = pd.DataFrame({
    #     "step": range(len(loss_history)),
    #     "loss": loss_history
    # })
    # df_loss.to_csv(
    #     os.path.join(output_dir, "loss_vs_epoch.csv")
    # )

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing device: {device}")

# Get parameters from META
bg_path = META['bg_path']
true_path = META['true_path']
bg_sep = ',' if META['bg_sep'] == '' else META['bg_sep']
output_base = META['output_path']
window_configs = META['window_configs']

# Track progress
job_counter = 0
total_jobs = len(csv_files) * len(window_configs)

# Process each CSV file
for csv_file in csv_files:
    csv_name = csv_file.stem
    csv_output_dir = os.path.join(output_base, csv_name)
    os.makedirs(csv_output_dir, exist_ok=True)
    
    print(f"\n{'#'*70}")
    print(f"# Processing CSV: {csv_file.name}")
    print(f"{'#'*70}")
    
    # Determine total timesteps in this CSV
    base_data = pd.read_csv(csv_file)
    total_timesteps = len(base_data.columns) - 1
    print(f"Total timesteps: {total_timesteps}")
    
    # Process each (N, M) configuration
    for window_size, step_size in window_configs:
        job_counter += 1
        
        config_name = f"N{window_size}_M{step_size}"
        config_output_dir = os.path.join(csv_output_dir, config_name)
        os.makedirs(config_output_dir, exist_ok=True)
        
        print(f"\n{'='*70}")
        print(f"[Job {job_counter}/{total_jobs}] CSV: {csv_name} | Config: {config_name}")
        print(f"{'='*70}")
        
        # Generate sliding windows for this configuration
        windows = []
        for start_idx in range(1, total_timesteps - window_size + 2, step_size):
            end_idx = start_idx + window_size - 1
            if end_idx <= total_timesteps:
                time_indices = list(range(start_idx, end_idx + 1))
                window_name = f"t{start_idx}-t{end_idx}"
                windows.append((time_indices, window_name))
        
        print(f"Generated {len(windows)} windows for this configuration")
        
        if len(windows) == 0:
            print(f"⚠ Skipping {config_name}: window_size ({window_size}) too large for data ({total_timesteps} timesteps)")
            continue
        
        # Store results for summary
        all_timestep_results = {}
        
        # Process each window
        for window_idx, (time_indices, window_name) in enumerate(windows, 1):
            window_output_dir = os.path.join(config_output_dir, window_name)
            
            try:
                print(f"  [{window_idx}/{len(windows)}] {window_name}...", end=" ", flush=True)
                
                # Train on this window
                results = train_single_window(
                    pos_path=str(csv_file),
                    bg_path=bg_path,
                    true_path=true_path,
                    output_dir=window_output_dir,
                    time_indices=time_indices,
                    window_name=window_name,
                    bg_sep=bg_sep,
                    scale_factor=SCALE_FACTOR,
                    num_epochs=1000,
                    device=device,
                    verbose=False  # Reduced verbosity for batch processing
                )
                
                # Save outputs
                save_outputs(results, window_output_dir, verbose=False)
                
                # Extract final g and h for each timestep
                final_g = results['g_final_list'][-1]
                final_h = results['h_final_list'][-1]
                
                for i, timestep in enumerate(time_indices):
                    if timestep not in all_timestep_results:
                        all_timestep_results[timestep] = {'g': [], 'h': []}
                    all_timestep_results[timestep]['g'].append(final_g[i])
                    all_timestep_results[timestep]['h'].append(final_h[i])
                
                print("✓")
                
            except Exception as e:
                print(f"✗ Error: {str(e)}")
                continue
        
        # Generate summary for this configuration
        print(f"\n  Generating summary for {config_name}...")
        
        timesteps = sorted(all_timestep_results.keys())
        summary_data = []
        
        for t in timesteps:
            g_vals = all_timestep_results[t]['g']
            h_vals = all_timestep_results[t]['h']
            
            g_mean = np.mean(g_vals)
            h_mean = np.mean(h_vals)
            
            summary_data.append({
                'timestep': t,
                'g': g_mean,
                'h': h_mean
            })
        
        # Save summary CSV
        df_summary = pd.DataFrame(summary_data)
        summary_csv_path = os.path.join(config_output_dir, "summary_g_h_vs_timestep.csv")
        df_summary.to_csv(summary_csv_path, index=False)
        
        # Plot summary
        plt.figure(figsize=(8, 5))
        plt.plot(df_summary['timestep'], df_summary['g'], 'o-', color='black', label='g (signal)', lw=2)
        plt.plot(df_summary['timestep'], df_summary['h'], 'o-', color='red', label='h (background)', lw=2)
        plt.xlabel('Timestep')
        plt.ylabel('Coefficient')
        plt.title(f'{csv_name} | {config_name} | Signal and Background Coefficients')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        
        summary_fig_path = os.path.join(config_output_dir, "summary_g_h_vs_timestep.png")
        plt.savefig(summary_fig_path, dpi=300)
        plt.close()
        
        print(f"  ✓ Summary saved for {config_name}")

print("\n" + "="*70)
print("ALL PROCESSING COMPLETED!")
print("="*70)
print(f"\nProcessed:")
print(f"  - {len(csv_files)} CSV files")
print(f"  - {len(window_configs)} window configurations per CSV")
print(f"  - Total: {total_jobs} combinations")
print(f"\nResults saved to: {output_base}")

# Post processing (independent code): generate final epoch's spectrum/similarity/R2/correlation vs timestep

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

"""
Post-processing script to generate summary CSVs from spectral decomposition results.
This script extracts final epoch data from all time windows and creates:
1. summary_extracted_spectrum_vs_timestep.csv
2. summary_metrics_vs_timestep.csv (R2, Cosine, Pearson)
"""

# ============================================================
# Configuration
# ============================================================
RESULTS_BASE_PATH = '/mnt/f/Jiaheng Cui/DNA RNA hybridization/Results/12192025-Yingchuan_scaled_code_time_dependent/Unnormalized_data/05212026-1E7-only_mean_norm-extraction'

# Set to None to process all CSVs, or specify a list like ['1.csv', '3.csv']
CSV_LIST = None

# Set to None to process all configs, or specify a list like ['N3_M3', 'N5_M2']
CONFIG_LIST = None

# ============================================================
# Main Processing
# ============================================================

def process_single_config(config_dir):
    """
    Process a single configuration directory (e.g., N3_M3) to generate summaries.
    
    Returns:
        True if successful, False otherwise
    """
    config_name = config_dir.name
    print(f"\n{'='*60}")
    print(f"Processing configuration: {config_name}")
    print(f"{'='*60}")
    
    # Find all time window subdirectories
    window_dirs = sorted([d for d in config_dir.iterdir() if d.is_dir() and d.name.startswith('t')])
    
    if len(window_dirs) == 0:
        print(f"⚠ No time window directories found in {config_name}")
        return False
    
    print(f"Found {len(window_dirs)} time windows")
    
    # Data structures to collect results
    timestep_data = {}  # key: timestep, value: dict with all data for that timestep
    
    # Process each window
    for window_dir in window_dirs:
        window_name = window_dir.name
        
        # Parse timesteps from window name (e.g., "t1-t3" -> [1, 2, 3])
        try:
            start_t, end_t = window_name.replace('t', '').split('-')
            timesteps = list(range(int(start_t), int(end_t) + 1))
        except:
            print(f"  ⚠ Could not parse timesteps from {window_name}, skipping")
            continue
        
        # Load the required CSVs
        extracted_csv = window_dir / "extracted_spectrum_vs_epoch.csv"
        metrics_csv = window_dir / "extracted_vs_true_metrics.csv"
        
        if not extracted_csv.exists() or not metrics_csv.exists():
            print(f"  ⚠ Missing CSV files in {window_name}, skipping")
            continue
        
        # Load data
        df_extracted = pd.read_csv(extracted_csv)
        df_metrics = pd.read_csv(metrics_csv)
        
        # Get final epoch column name (last column that starts with "Epoch_")
        epoch_cols = [col for col in df_extracted.columns if col.startswith('Epoch_')]
        if len(epoch_cols) == 0:
            print(f"  ⚠ No epoch columns found in {window_name}, skipping")
            continue
        
        final_epoch_col = epoch_cols[-1]
        
        # Extract final epoch spectrum (should be one spectrum for all timesteps in this window)
        final_spectrum = df_extracted[final_epoch_col].values
        wavenumbers = df_extracted['Wavenumbers'].values
        
        # Extract final epoch metrics (should be 3 rows: R2, Cosine, Pearson)
        # The metrics CSV has metrics as rows and epochs as columns
        metrics_epoch_cols = [col for col in df_metrics.columns if col.startswith('Epoch_')]
        if len(metrics_epoch_cols) == 0:
            print(f"  ⚠ No epoch columns found in metrics for {window_name}, skipping")
            continue
        
        final_metrics_col = metrics_epoch_cols[-1]
        final_r2 = df_metrics.loc[df_metrics.iloc[:, 0] == 'R2', final_metrics_col].values[0]
        final_cosine = df_metrics.loc[df_metrics.iloc[:, 0] == 'Cosine', final_metrics_col].values[0]
        final_pearson = df_metrics.loc[df_metrics.iloc[:, 0] == 'Pearson', final_metrics_col].values[0]
        
        # Store data for each timestep
        for timestep in timesteps:
            if timestep not in timestep_data:
                timestep_data[timestep] = {
                    'spectrum': [],
                    'r2': [],
                    'cosine': [],
                    'pearson': []
                }
            
            timestep_data[timestep]['spectrum'].append(final_spectrum)
            timestep_data[timestep]['r2'].append(final_r2)
            timestep_data[timestep]['cosine'].append(final_cosine)
            timestep_data[timestep]['pearson'].append(final_pearson)
        
        print(f"  ✓ Processed {window_name}: timesteps {timesteps}")
    
    if len(timestep_data) == 0:
        print(f"⚠ No valid data collected for {config_name}")
        return False
    
    # ============================================================
    # Generate Summary CSVs
    # ============================================================
    
    # 1. Summary: Extracted Spectrum vs Timestep
    # Average spectra if a timestep appears in multiple windows
    timesteps_sorted = sorted(timestep_data.keys())
    
    spectrum_data = {'Wavenumbers': wavenumbers}
    for t in timesteps_sorted:
        # Average all spectra for this timestep
        spectra_array = np.array(timestep_data[t]['spectrum'])
        avg_spectrum = np.mean(spectra_array, axis=0)
        spectrum_data[f't{t}'] = avg_spectrum
    
    df_spectrum_summary = pd.DataFrame(spectrum_data)
    spectrum_csv_path = config_dir / "summary_extracted_spectrum_vs_timestep.csv"
    df_spectrum_summary.to_csv(spectrum_csv_path, index=False)
    print(f"\n✓ Saved: {spectrum_csv_path.name}")
    
    # 2. Summary: Metrics vs Timestep
    metrics_data = {
        'timestep': timesteps_sorted,
        'R2': [],
        'Cosine': [],
        'Pearson': []
    }
    
    for t in timesteps_sorted:
        # Average metrics if a timestep appears in multiple windows
        metrics_data['R2'].append(np.mean(timestep_data[t]['r2']))
        metrics_data['Cosine'].append(np.mean(timestep_data[t]['cosine']))
        metrics_data['Pearson'].append(np.mean(timestep_data[t]['pearson']))
    
    df_metrics_summary = pd.DataFrame(metrics_data)
    metrics_csv_path = config_dir / "summary_metrics_vs_timestep.csv"
    df_metrics_summary.to_csv(metrics_csv_path, index=False)
    print(f"✓ Saved: {metrics_csv_path.name}")
    
    print(f"\n✓ Successfully completed {config_name}")
    return True


# ============================================================
# Main Execution
# ============================================================

results_base = Path(RESULTS_BASE_PATH)

if not results_base.exists():
    raise ValueError(f"Results base path does not exist: {RESULTS_BASE_PATH}")

# Get list of CSV folders to process
if CSV_LIST is None:
    csv_folders = sorted([d for d in results_base.iterdir() if d.is_dir()])
else:
    csv_folders = [results_base / csv_name.replace('.csv', '') for csv_name in CSV_LIST]
    csv_folders = [f for f in csv_folders if f.exists()]

if len(csv_folders) == 0:
    raise ValueError("No CSV folders found to process!")

print(f"Found {len(csv_folders)} CSV folders to process")

# Process each CSV folder
total_processed = 0
total_failed = 0

for csv_folder in csv_folders:
    print(f"\n{'#'*70}")
    print(f"CSV Folder: {csv_folder.name}")
    print(f"{'#'*70}")
    
    # Get list of configuration folders
    if CONFIG_LIST is None:
        config_dirs = sorted([d for d in csv_folder.iterdir() if d.is_dir() and d.name.startswith('N')])
    else:
        config_dirs = [csv_folder / config_name for config_name in CONFIG_LIST]
        config_dirs = [d for d in config_dirs if d.exists()]
    
    if len(config_dirs) == 0:
        print(f"⚠ No configuration folders found in {csv_folder.name}")
        continue
    
    print(f"Found {len(config_dirs)} configurations")
    
    # Process each configuration
    for config_dir in config_dirs:
        success = process_single_config(config_dir)
        if success:
            total_processed += 1
        else:
            total_failed += 1

print(f"\n{'='*70}")
print(f"POST-PROCESSING COMPLETED")
print(f"{'='*70}")
print(f"Successfully processed: {total_processed} configurations")
print(f"Failed: {total_failed} configurations")